# Lesson 3: From Cypher to Native Projection

**Duration:** ~10 minutes  
**Module:** GDS with Python  
**Dataset:** Movies (Actors, Movies, Genres, Users)

## What You'll Learn

By the end of this notebook, you'll be able to:

- Translate Cypher projections to Python syntax
- Configure node and relationship properties in projections
- Set relationship orientation (undirected, reverse)
- Handle missing properties with default values
- Choose between `gds.graph.project()` and `gds.graph.cypher.project()`

## Setup: Connect to Neo4j

First, let's establish our connection.

In [10]:
import os
import pandas as pd
from IPython.display import display
from graphdatascience import GraphDataScience
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

uri = os.getenv('NEO4J_URI')
username = os.getenv('NEO4J_USERNAME')
password = os.getenv('NEO4J_PASSWORD')

# Create GDS client
gds = GraphDataScience(uri, auth=(username, password))

print(f"Connected to GDS version: {gds.version()}")

Connected to GDS version: 2.25.0


In [11]:
# Verify our data is loaded from Lesson 2
result = gds.run_cypher("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(*) AS count
    ORDER BY count DESC
""")
print("Node counts:")
display(result)

Node counts:


,label,count
0,Actor,15443
1,Movie,9122
2,User,671
3,Genre,20


In [14]:
# Clean up any existing projections from previous runs
for graph_name in gds.graph.list()["graphName"].tolist():
    gds.graph.drop(graph_name)
    print(f"Dropped: {graph_name}")
print("Ready to start!")

Ready to start!


## The Same Concepts, Different Syntax

You've been using Cypher projections throughout this course. The Python client uses the same concepts:

- Select nodes by label
- Select relationships by type
- Configure properties and orientation
- Create an in-memory projection

The difference is syntax: Python dictionaries instead of Cypher.

## Basic Projection: Cypher vs Python

**Cypher (what you already know):**

```cypher
MATCH (source:Movie)-[r:IN_GENRE]->(target:Genre)
WITH gds.graph.project('movies-genres', source, target) AS g
RETURN g.graphName, g.nodeCount
```

**Python equivalent:**

In [15]:
# Basic projection - labels and type as simple arguments
G, result = gds.graph.project(
    "movies-genres",      # Graph name
    ["Movie", "Genre"],   # Node labels
    "IN_GENRE"            # Relationship type
)

print(f"Graph: {G.name()}")
print(f"Nodes: {G.node_count()}")
print(f"Relationships: {G.relationship_count()}")

Graph: movies-genres
Nodes: 9142
Relationships: 20338


In [16]:
# The result metadata - same fields as Cypher projection
print(f"graphName: {result['graphName']}")
print(f"nodeCount: {result['nodeCount']}")
print(f"relationshipCount: {result['relationshipCount']}")
print(f"projectMillis: {result['projectMillis']}")

graphName: movies-genres
nodeCount: 9142
relationshipCount: 20338
projectMillis: 20


In [17]:
# Clean up before next example
G.drop()

graphName                                                    movies-genres
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                             9142
relationshipCount                                                    20338
configuration            {'relationshipProjection': {'IN_GENRE': {'orie...
density                                                           0.000243
creationTime                           2026-02-04T14:17:38.454330000+00:00
modificationTime                       2026-02-04T14:17:38.454330000+00:00
schema                   {'graphProperties': {}, 'relationships': {'IN_...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'IN_...
Name: 0, dtype: object

## Multiple Labels and Types

**Cypher:**

```cypher
MATCH (source)-[r:ACTED_IN|RATED]->(target:Movie)
WHERE source:Actor OR source:User
WITH gds.graph.project('movie-interactions', source, target) AS g
RETURN g
```

**Python:**

In [18]:
# Multiple labels and relationship types as lists
G, _ = gds.graph.project(
    "movie-interactions",
    ["Actor", "User", "Movie"],  # Multiple labels
    ["ACTED_IN", "RATED"]        # Multiple relationship types
)

print(f"Node labels: {G.node_labels()}")
print(f"Relationship types: {G.relationship_types()}")
print(f"Total nodes: {G.node_count()}")
print(f"Total relationships: {G.relationship_count()}")

Node labels: ['Movie', 'User', 'Actor']
Relationship types: ['ACTED_IN', 'RATED']
Total nodes: 25236
Total relationships: 135913


In [19]:
G.drop()

graphName                                               movie-interactions
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                            25236
relationshipCount                                                   135913
configuration            {'relationshipProjection': {'ACTED_IN': {'orie...
density                                                           0.000213
creationTime                           2026-02-04T14:17:44.532945000+00:00
modificationTime                       2026-02-04T14:17:44.532945000+00:00
schema                   {'graphProperties': {}, 'relationships': {'ACT...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'ACT...
Name: 0, dtype: object

## Adding Node Properties

**Cypher:**

```cypher
MATCH (source:Actor)-[r:ACTED_IN]->(target:Movie)
WITH gds.graph.project(
  'actors-movies-props',
  source, target,
  { 
    sourceNodeProperties: source { .born }, 
    targetNodeProperties: target { .year, .imdbRating }
  }
) AS g
RETURN g
```

**Python:** When you need properties, labels become dictionary keys.

In [20]:
# Node properties - labels become dictionary keys
G, _ = gds.graph.project(
    "actors-movies-props",
    {
        "Actor": {},
        "Movie": {
            "properties": ["year"]
        }
    },
    "ACTED_IN"
)

print(f"Movie properties: {G.node_properties('Movie')}")

Movie properties: ['year']


In [21]:
G.drop()

graphName                                              actors-movies-props
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                            24565
relationshipCount                                                    35910
configuration            {'relationshipProjection': {'ACTED_IN': {'orie...
density                                                            0.00006
creationTime                           2026-02-04T14:17:48.881809000+00:00
modificationTime                       2026-02-04T14:17:48.881809000+00:00
schema                   {'graphProperties': {}, 'relationships': {'ACT...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'ACT...
Name: 0, dtype: object

## Relationship Properties

**Cypher:**

```cypher
MATCH (source:User)-[r:RATED]->(target:Movie)
WITH gds.graph.project(
  'user-ratings',
  source, target,
  { relationshipProperties: r { .rating } }
) AS g
RETURN g
```

**Python:** Same pattern - relationship types become dictionary keys when you need configuration.

In [22]:
# Relationship properties - type becomes a dictionary key
G, _ = gds.graph.project(
    "user-ratings",
    ["User", "Movie"],
    {
        "RATED": {
            "properties": ["rating"]
        }
    }
)

print(f"Relationship types: {G.relationship_types()}")
print(f"Nodes: {G.node_count()}")
print(f"Relationships: {G.relationship_count()}")

Relationship types: ['RATED']
Nodes: 9793
Relationships: 100003


In [23]:
G.drop()

graphName                                                     user-ratings
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                             9793
relationshipCount                                                   100003
configuration            {'relationshipProjection': {'RATED': {'orienta...
density                                                           0.001043
creationTime                           2026-02-04T14:17:53.611325000+00:00
modificationTime                       2026-02-04T14:17:53.611325000+00:00
schema                   {'graphProperties': {}, 'relationships': {'RAT...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'RAT...
Name: 0, dtype: object

## Undirected Relationships

**Cypher:**

```cypher
MATCH (source:Actor)-[r:ACTED_IN]->(target:Movie)
WITH gds.graph.project(
  'actors-movies-undirected',
  source, target,
  {},
  { undirectedRelationshipTypes: ['ACTED_IN'] }
) AS g
RETURN g
```

**Python:** Use `orientation` in the relationship configuration.

In [24]:
# Undirected relationships
G, _ = gds.graph.project(
    "actors-movies-undirected",
    ["Actor", "Movie"],
    {
        "ACTED_IN": {
            "orientation": "UNDIRECTED"
        }
    }
)

print(f"Relationships: {G.relationship_count()}")
print("(Notice: count is doubled because each edge is bidirectional)")

Relationships: 71820
(Notice: count is doubled because each edge is bidirectional)


**Orientation options:**

- `NATURAL` — Keep original direction (default)
- `REVERSE` — Flip all directions
- `UNDIRECTED` — Treat as bidirectional

In [25]:
G.drop()

graphName                                         actors-movies-undirected
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                            24565
relationshipCount                                                    71820
configuration            {'relationshipProjection': {'ACTED_IN': {'orie...
density                                                           0.000119
creationTime                           2026-02-04T14:17:58.081735000+00:00
modificationTime                       2026-02-04T14:17:58.081735000+00:00
schema                   {'graphProperties': {}, 'relationships': {'ACT...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'ACT...
Name: 0, dtype: object

## Default Values for Missing Properties

Some nodes might be missing properties. Use `defaultValue` to handle nulls:

In [26]:
# Check how many actors are missing 'born' property
result = gds.run_cypher("""
    MATCH (a:Actor)
    RETURN 
        count(*) AS total,
        sum(CASE WHEN a.born IS NULL THEN 1 ELSE 0 END) AS missing_born
""")
display(result)

,total,missing_born
0,15443,15443


In [27]:
# Project with default values for missing properties
G, _ = gds.graph.project(
    "with-defaults",
    {
        "Actor": {
            "properties": {
                "born": {
                    "property": "born",
                    "defaultValue": 1900  # We can use this default value to handle nulls
                }
            }
        },
        "Movie": {
            "properties": {
                "year": {"defaultValue": 1900},
                "imdbRating": {"defaultValue": 0.0},
                "runtime": {"defaultValue": 90}
            }
        }
    },
    {
        "ACTED_IN": {
            "orientation": "UNDIRECTED"
        }
    }
)

print(f"Actor properties: {G.node_properties('Actor')}")
print(f"Movie properties: {G.node_properties('Movie')}")
print("\nNodes with missing properties now use default values!")

Actor properties: ['born']
Movie properties: ['runtime', 'imdbRating', 'year']

Nodes with missing properties now use default values!


In [28]:
G.drop()

graphName                                                    with-defaults
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                            24565
relationshipCount                                                    71820
configuration            {'relationshipProjection': {'ACTED_IN': {'orie...
density                                                           0.000119
creationTime                           2026-02-04T14:18:04.163304000+00:00
modificationTime                       2026-02-04T14:18:04.163304000+00:00
schema                   {'graphProperties': {}, 'relationships': {'ACT...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'ACT...
Name: 0, dtype: object

## Monopartite Transformations

Creating an Actor-to-Actor network through shared Movies requires a complex MATCH pattern.

**Cypher:**

```cypher
MATCH (source:Actor)-[:ACTED_IN]->(:Movie)<-[:ACTED_IN]-(target:Actor)
WITH gds.graph.project('actor-collab', source, target) AS g
RETURN g
```

`gds.graph.project()` only handles direct label/type projections. For pattern-based transformations, use `gds.graph.cypher.project()`:

In [29]:
# Monopartite transformation using Cypher projection
G, result = gds.graph.cypher.project(
    """
    MATCH (source:Actor)-[:ACTED_IN]->(:Movie)<-[:ACTED_IN]-(target:Actor)
    RETURN gds.graph.project($graph_name, source, target)
    """,
    graph_name='actor-collab'
)

print(f"Actor collaboration network:")
print(f"  Actors: {G.node_count()}")
print(f"  Collaborations: {G.relationship_count()}")

Actor collaboration network:
  Actors: 15408
  Collaborations: 107170


In [30]:
# Run an algorithm on the collaboration network
df = gds.degree.stream(G)
print("Most connected actors (by collaborations):")
display(df.nlargest(5, "score"))

Most connected actors (by collaborations):


,nodeId,score
215,11285,168.0
726,13807,147.0
776,12945,135.0
876,15423,131.0
220,10289,120.0


In [31]:
G.drop()

graphName                                                     actor-collab
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                            15408
relationshipCount                                                   107170
configuration            {'jobId': 'jid-e4305159-f0ac-4b64-ac00-5473554...
density                                                           0.000451
creationTime                           2026-02-04T14:18:08.016291000+00:00
modificationTime                       2026-02-04T14:18:08.016291000+00:00
schema                   {'graphProperties': {}, 'relationships': {'__A...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'__A...
Name: 0, dtype: object

## Alternative: run_cypher + graph.get()

You can also run Cypher projection directly and retrieve the Graph object afterwards:

In [32]:
# Run the exact Cypher syntax you already know
result = gds.run_cypher("""
    MATCH (source:Actor)-[:ACTED_IN]->(:Movie)<-[:ACTED_IN]-(target:Actor)
    WITH gds.graph.project('actor-collab-v2', source, target) AS g
    RETURN g.graphName, g.nodeCount, g.relationshipCount
""")
display(result)

# Then retrieve the Graph object
G = gds.graph.get("actor-collab-v2") # We're just using v2 here as we haven't dropped the previous graph yet
print(f"\nRetrieved graph: {G.name()}")
print(f"Node count: {G.node_count()}")

,g.graphName,g.nodeCount,g.relationshipCount
0,actor-collab-v2,15408,107170



Retrieved graph: actor-collab-v2
Node count: 15408


In [33]:
G.drop()

graphName                                                  actor-collab-v2
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                            15408
relationshipCount                                                   107170
configuration            {'jobId': 'jid-dfd2ebe3-613c-488b-a5b3-c194884...
density                                                           0.000451
creationTime                           2026-02-04T14:18:12.554526000+00:00
modificationTime                       2026-02-04T14:18:12.554526000+00:00
schema                   {'graphProperties': {}, 'relationships': {'__A...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'__A...
Name: 0, dtype: object

## Complex Projection: Full Movies Graph

Let's project the complete Movies graph with all node types, relationship types, and properties:

In [34]:
# Full Movies graph projection
G, result = gds.graph.project(
    "movies-full",
    {
        "Actor": {
            "properties": {
                "born": {"defaultValue": 1900}
            }
        },
        "User": {},
        "Movie": {
            "properties": {
                "imdbRating": {"defaultValue": 0.0},
                "year": {"defaultValue": 1900}
            }
        },
        "Genre": {}
    },
    {
        "ACTED_IN": {},
        "RATED": {
            "properties": {
                "rating": {"defaultValue": 0.0}
            }
        },
        "IN_GENRE": {}
    }
)

print("Full Movies Graph:")
print(f"  Node labels: {G.node_labels()}")
print(f"  Relationship types: {G.relationship_types()}")
print(f"  Total nodes: {G.node_count()}")
print(f"  Total relationships: {G.relationship_count()}")
print(f"  Memory usage: {G.memory_usage()}")

Full Movies Graph:
  Node labels: ['Movie', 'User', 'Genre', 'Actor']
  Relationship types: ['IN_GENRE', 'ACTED_IN', 'RATED']
  Total nodes: 25256
  Total relationships: 156251
  Memory usage: 11106 KiB


In [35]:
# Check properties on each label
for label in G.node_labels():
    props = G.node_properties(label)
    print(f"{label}: {props if props else '(no properties)'}")

Movie: ['year', 'imdbRating']
User: (no properties)
Genre: (no properties)
Actor: ['born']


In [36]:
G.drop()

graphName                                                      movies-full
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                            25256
relationshipCount                                                   156251
configuration            {'relationshipProjection': {'IN_GENRE': {'orie...
density                                                           0.000245
creationTime                           2026-02-04T14:18:16.740878000+00:00
modificationTime                       2026-02-04T14:18:16.740878000+00:00
schema                   {'graphProperties': {}, 'relationships': {'IN_...
schemaWithOrientation    {'graphProperties': {}, 'relationships': {'IN_...
Name: 0, dtype: object

## Translation Reference

| **Cypher** | **Python** |
|------------|------------|
| `MATCH (source:Label)` | `"Label"` or `{"Label": {}}` |
| `sourceNodeProperties: source { .prop }` | `{"Label": {"properties": ["prop"]}}` |
| `relationshipProperties: r { .weight }` | `{"TYPE": {"properties": ["weight"]}}` |
| `undirectedRelationshipTypes: ['TYPE']` | `{"TYPE": {"orientation": "UNDIRECTED"}}` |
| Complex MATCH patterns | Use `gds.graph.cypher.project()` |

## When to Use Each Method

| **Method** | **Use when** |
|------------|-------------|
| `gds.graph.project()` | Projecting by labels and types directly |
| `gds.graph.cypher.project()` | Complex patterns, aggregation, filtering with WHERE |

Most projections work with `gds.graph.project()`. Use `gds.graph.cypher.project()` when you need the full flexibility of Cypher patterns—like creating monopartite transformations through intermediate nodes.

## Summary

You've learned to translate Cypher projections to Python:

| Concept | Simple Syntax | With Configuration |
|---------|---------------|--------------------|
| Node labels | `"Label"` or `["Label1", "Label2"]` | `{"Label": {"properties": [...]}}` |
| Relationship types | `"TYPE"` or `["TYPE1", "TYPE2"]` | `{"TYPE": {"orientation": "...", "properties": [...]}}` |

**Key insights:**
- Simple projections use strings or lists
- Add configuration by converting to dictionaries
- Use `defaultValue` to handle missing properties
- Use `gds.graph.cypher.project()` for complex MATCH patterns
- Use `gds.graph.get()` to retrieve existing projections

### Next Lesson

In Lesson 4, you'll learn to run algorithms and chain them together in Python workflows.

In [37]:
# Clean up any remaining projections
for graph_name in gds.graph.list()["graphName"].tolist():
    gds.graph.drop(graph_name)
    print(f"Dropped: {graph_name}")